In [1]:
import pandas as pd
import streamlit as st
import requests
import json
import matplotlib.pyplot as plt
import pickle
import shap

Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)


In [2]:
echantillon_clients = pd.read_csv("echantillon_clients.csv", index_col="SK_ID_CURR")

In [3]:
echantillon_clients

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,...,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,CC_NAME_CONTRACT_STATUS_Sent proposal_SUM,CC_NAME_CONTRACT_STATUS_Sent proposal_VAR,CC_NAME_CONTRACT_STATUS_Signed_MIN,CC_NAME_CONTRACT_STATUS_Signed_MAX,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_Signed_SUM,CC_NAME_CONTRACT_STATUS_Signed_VAR,CC_COUNT,threshold
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
210332,0.0,1.0,0.0,0.0,540000.0,675000.0,70879.5,675000.0,0.026392,-11246.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0,0.517826
236766,1.0,0.0,0.0,0.0,135000.0,817560.0,30951.0,675000.0,0.031329,-20595.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.517826
174607,1.0,0.0,0.0,1.0,225000.0,601470.0,30838.5,450000.0,0.005084,-15203.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.0,0.517826
144965,1.0,0.0,0.0,0.0,112500.0,360000.0,22153.5,360000.0,0.026392,-20534.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.517826
216121,0.0,1.0,0.0,0.0,225000.0,900000.0,29034.0,900000.0,0.026392,-20589.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.517826
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103460,1.0,0.0,0.0,0.0,360000.0,295776.0,18225.0,234000.0,0.020246,-22869.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.517826
434205,0.0,1.0,0.0,1.0,135000.0,218938.5,22563.0,189000.0,0.010556,-24784.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.517826
382113,1.0,1.0,0.0,2.0,427500.0,450000.0,22500.0,225000.0,0.024610,-14645.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0,0.517826


In [4]:
echantillon_clients = echantillon_clients.drop(columns=["threshold"])
echantillon_clients

,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,...,CC_NAME_CONTRACT_STATUS_Sent proposal_MAX,CC_NAME_CONTRACT_STATUS_Sent proposal_MEAN,CC_NAME_CONTRACT_STATUS_Sent proposal_SUM,CC_NAME_CONTRACT_STATUS_Sent proposal_VAR,CC_NAME_CONTRACT_STATUS_Signed_MIN,CC_NAME_CONTRACT_STATUS_Signed_MAX,CC_NAME_CONTRACT_STATUS_Signed_MEAN,CC_NAME_CONTRACT_STATUS_Signed_SUM,CC_NAME_CONTRACT_STATUS_Signed_VAR,CC_COUNT
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,
210332,0.0,1.0,0.0,0.0,540000.0,675000.0,70879.5,675000.0,0.026392,-11246.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0
236766,1.0,0.0,0.0,0.0,135000.0,817560.0,30951.0,675000.0,0.031329,-20595.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0
174607,1.0,0.0,0.0,1.0,225000.0,601470.0,30838.5,450000.0,0.005084,-15203.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.0
144965,1.0,0.0,0.0,0.0,112500.0,360000.0,22153.5,360000.0,0.026392,-20534.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0
216121,0.0,1.0,0.0,0.0,225000.0,900000.0,29034.0,900000.0,0.026392,-20589.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103460,1.0,0.0,0.0,0.0,360000.0,295776.0,18225.0,234000.0,0.020246,-22869.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0
434205,0.0,1.0,0.0,1.0,135000.0,218938.5,22563.0,189000.0,0.010556,-24784.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0
382113,1.0,1.0,0.0,2.0,427500.0,450000.0,22500.0,225000.0,0.024610,-14645.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.0


In [16]:
data_client = echantillon_clients.loc[echantillon_clients.index == 210332].to_dict()

In [17]:
data_client

{'CODE_GENDER': {210332: 0.0},
 'FLAG_OWN_CAR': {210332: 1.0},
 'FLAG_OWN_REALTY': {210332: 0.0},
 'CNT_CHILDREN': {210332: 0.0},
 'AMT_INCOME_TOTAL': {210332: 540000.0},
 'AMT_CREDIT': {210332: 675000.0},
 'AMT_ANNUITY': {210332: 70879.5},
 'AMT_GOODS_PRICE': {210332: 675000.0},
 'REGION_POPULATION_RELATIVE': {210332: 0.026392},
 'DAYS_BIRTH': {210332: -11246.0},
 'DAYS_EMPLOYED': {210332: -2539.0},
 'DAYS_REGISTRATION': {210332: -5469.0},
 'DAYS_ID_PUBLISH': {210332: -3405.0},
 'OWN_CAR_AGE': {210332: 3.0},
 'FLAG_MOBIL': {210332: 1.0},
 'FLAG_EMP_PHONE': {210332: 1.0},
 'FLAG_WORK_PHONE': {210332: 1.0},
 'FLAG_CONT_MOBILE': {210332: 1.0},
 'FLAG_PHONE': {210332: 1.0},
 'FLAG_EMAIL': {210332: 1.0},
 'CNT_FAM_MEMBERS': {210332: 2.0},
 'REGION_RATING_CLIENT': {210332: 2.0},
 'REGION_RATING_CLIENT_W_CITY': {210332: 2.0},
 'HOUR_APPR_PROCESS_START': {210332: 20.0},
 'REG_REGION_NOT_LIVE_REGION': {210332: 0.0},
 'REG_REGION_NOT_WORK_REGION': {210332: 0.0},
 'LIVE_REGION_NOT_WORK_REGION': 

In [21]:
ID = 210332

In [31]:
data_json = {'data': ID}

In [32]:
response = requests.request(
        method='POST', url="http://127.0.0.1:8000/invocations", json=data_json)

In [33]:
response

<Response [422]>